In [2]:
from datasets import load_dataset

ds = load_dataset("jjzha/skillspan", download_mode="force_redownload")

Generating test split: 100%|██████████| 3569/3569 [00:00<00:00, 663746.33 examples/s]


In [3]:
ds.items()

dict_items([('train', Dataset({
    features: ['idx', 'tokens', 'tags_skill', 'tags_knowledge', 'source'],
    num_rows: 4800
})), ('validation', Dataset({
    features: ['idx', 'tokens', 'tags_skill', 'tags_knowledge', 'source'],
    num_rows: 3174
})), ('test', Dataset({
    features: ['idx', 'tokens', 'tags_skill', 'tags_knowledge', 'source'],
    num_rows: 3569
}))])

In [4]:
import pandas as pd

train_df = pd.DataFrame(ds['train'])
val_df = pd.DataFrame(ds['validation'])
test_df = pd.DataFrame(ds['test'])

In [5]:
train_df.head()

,idx,tokens,tags_skill,tags_knowledge,source
0,1,"[Senior, QA, Engineer, (, m/f/d, ), <ORGANIZAT...","[O, O, O, O, O, O, O]","[O, O, O, O, O, O, O]",tech
1,1,"[<ADDRESS>, <ADDRESS>, <ADDRESS>, <ADDRESS>, <...","[O, O, O, O, O]","[O, O, O, O, O]",tech
2,1,"[Date, posted:, 2021-07-14]","[O, O, O]","[O, O, O]",tech
3,1,"[Likes:, 0, Dislikes:, 0, Love:, 0]","[O, O, O, O, O, O]","[O, O, O, O, O, O]",tech
4,1,"[Job, description:]","[O, O]","[O, O]",tech


In [6]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(4800, 5)
(3174, 5)
(3569, 5)


In [7]:
train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

df = pd.concat([train_df, val_df, test_df], ignore_index = True)
df

,idx,tokens,tags_skill,tags_knowledge,source,split
0,1,"[Senior, QA, Engineer, (, m/f/d, ), <ORGANIZAT...","[O, O, O, O, O, O, O]","[O, O, O, O, O, O, O]",tech,train
1,1,"[<ADDRESS>, <ADDRESS>, <ADDRESS>, <ADDRESS>, <...","[O, O, O, O, O]","[O, O, O, O, O]",tech,train
2,1,"[Date, posted:, 2021-07-14]","[O, O, O]","[O, O, O]",tech,train
3,1,"[Likes:, 0, Dislikes:, 0, Love:, 0]","[O, O, O, O, O, O]","[O, O, O, O, O, O]",tech,train
4,1,"[Job, description:]","[O, O]","[O, O]",tech,train
...,...,...,...,...,...,...
11538,65,"[You, can, read, about, the, recruitment, proc...","[O, O, O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O, O, O]",house,test
11539,65,"[Assessment:, The, selected, applications, wil...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O]",house,test
11540,65,"[242, of, 13, March, 2012, on, the, Appointmen...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",house,test
11541,65,"[The, Assessment, Committee, makes, a, non-pri...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",house,test


In [8]:
source_split = df['source'].value_counts()
print("sentence split: ", source_split)

sentence split:  source
tech     7575
house    3968
Name: count, dtype: int64


In [9]:
def count_spans(col):
    b_tag = 0

    for row in col:
        if row.startswith('B'):
            b_tag += 1

    return b_tag

In [10]:
num_knowledge = df['tags_knowledge'].apply(count_spans).sum()

num_skills = df['tags_skill'].apply(count_spans).sum()

print(num_knowledge)
print(num_skills)

5236
4381


In [11]:
knowledge_split = df['tags_knowledge'].apply(count_spans).groupby(df['split']).sum()

skills_split = df['tags_skill'].apply(count_spans).groupby(df['split']).sum()

print("Count spans for knowledge tags - grouped by split")
print(knowledge_split)

print("Count spans for skill tags - grouped by split")
print(skills_split)

Count spans for knowledge tags - grouped by split
split
test          1174
train         2969
validation    1093
Name: tags_knowledge, dtype: int64
Count spans for skill tags - grouped by split
split
test          1090
train         2221
validation    1070
Name: tags_skill, dtype: int64


In [12]:
knowledge_source = df['tags_knowledge'].apply(count_spans).groupby(df['source']).sum()

skills_source = df['tags_skill'].apply(count_spans).groupby(df['source']).sum()

print("Count spans for knowledge tags - grouped by source")
print(knowledge_source)

print("Count spans for skill tags - grouped by source")
print(skills_source)

Count spans for knowledge tags - grouped by source
source
house    1436
tech     3800
Name: tags_knowledge, dtype: int64
Count spans for skill tags - grouped by source
source
house    2160
tech     2221
Name: tags_skill, dtype: int64


In [13]:
crosstab = pd.crosstab(df['split'], df['source'])
print(crosstab)

source      house  tech
split                  
test         1283  2286
train        1667  3133
validation   1018  2156


In [14]:
df["num_skills"] = df['tags_skill'].apply(count_spans)
df["num_knowledge"] = df['tags_knowledge'].apply(count_spans)

df_no_span = df[(df['num_skills'] == 0) &(df['num_knowledge'] == 0)]
print(f"There are {len(df_no_span)} sentences with zero spans")

There are 8071 sentences with zero spans


### Function to check if any token has both a skill and knowledge tag (not 'O')

In [15]:
def has_overlap(row):
    for skill, knowledge in zip(row['tags_skill'], row['tags_knowledge']):
        if skill != 'O' and knowledge != 'O':
            return True
    return False

In [16]:
df['has_overlap'] = df.apply(has_overlap, axis = 1)
overlap_df = df[df['has_overlap']]

print(f"There are {len(overlap_df)} sentences with overlapped tags!")

There are 134 sentences with overlapped tags!


In [17]:
overlap_df.head()

,idx,tokens,tags_skill,tags_knowledge,source,split,num_skills,num_knowledge,has_overlap
23,1,"[You, will, be, working, in, an, end-to-end, c...","[O, O, O, O, O, O, O, O, O, O, O, O, B, I, I, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",tech,train,1,1,True
26,1,"[Work, hands-on, together, with, the, other, e...","[B, I, O, O, O, O, O, O, O, O, O, O, O, B, I, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",tech,train,3,1,True
430,6,"[Work, as, part, of, a, dedicated, backend, te...","[O, O, O, O, O, O, O, O, O, B, I, I, I, I, I, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",tech,train,1,1,True
592,8,"[•, Must, have, experience, developing, Java, ...","[O, O, O, O, B, I, I, I, O, O, O, O, O, O, O, ...","[O, O, O, O, O, B, O, O, O, O, B, I, I, I, O, ...",tech,train,1,5,True
596,8,"[•, Understanding, of, the, mobile, developmen...","[O, O, O, O, O, O, O, O, O, O, O, B, I, I, I, I]","[O, O, O, O, B, I, I, O, O, O, O, O, B, I, I, I]",tech,train,1,2,True


### Function to check if there are any broken span ("I" after 'O')

In [18]:
def check_broken_span(row):
    if len(row) > 0 and row[0].startswith('I'):
        return True
    for i in range(1, len(row)):
        if row[i].startswith('I') and row[i-1].startswith('O'):
            return True
    return False

In [19]:
def fix_broken_span(row):
    if len(row) == 0:
        return row

    if row[0].startswith("I"):
        row[0] = 'B' + row[0][1:]

    for i in range(1, len(row)):
        if row[i].startswith('I') and row[i-1].startswith('O'):
            row[i] = "B" + row[i][1:]

    return row

In [20]:
df['broken_span_skill'] = df['tags_skill'].apply(check_broken_span)
df['broken_span_knowledge'] = df['tags_knowledge'].apply(check_broken_span)


broken_spans_df = df[df['broken_span_skill'] | df['broken_span_knowledge']]
print(f"There are {len(broken_spans_df)} sentences with broken spans!")
broken_spans_df

There are 1 sentences with broken spans!


,idx,tokens,tags_skill,tags_knowledge,source,split,num_skills,num_knowledge,has_overlap,broken_span_skill,broken_span_knowledge
8951,13,"[Experience, with, agile, approaches, to, soft...","[O, O, O, O, O, O, I, I, I]","[O, O, B, I, I, I, O, O, O]",tech,test,0,1,False,True,False


In [21]:
print("Fix the broken spans!")
df['tags_skill'] = df['tags_skill'].apply(fix_broken_span)
df['tags_knowledge'] = df['tags_knowledge'].apply(fix_broken_span)

#verify the fix
df['broken_span_skill'] = df['tags_skill'].apply(check_broken_span)
df['broken_span_knowledge'] = df['tags_knowledge'].apply(check_broken_span)
broken_spans_df = df[df['broken_span_skill'] | df['broken_span_knowledge']]
print(f"There are {len(broken_spans_df)} sentences with broken spans!")

df['num_skills'] = df['tags_skill'].apply(count_spans)
df['num_knowledge'] = df['tags_knowledge'].apply(count_spans)

print(df['num_skills'].sum())

Fix the broken spans!
There are 0 sentences with broken spans!
4382


Row 8951 (test set): skill tags started with I and had no B. Converted the leading I to B. This is the only broken span in the dataset.

### Check token distribution

In [22]:
df['tokens_length'] = df['tokens'].apply(len)

print(df['tokens_length'].describe())

count    11543.000000
mean        15.272979
std         20.944938
min          1.000000
25%          3.000000
50%         11.000000
75%         21.000000
max        748.000000
Name: tokens_length, dtype: float64


**Summary of the dataset:**
- On average, there are 15 tokens in a sentence
- The median is 11 tokens in a sentence
- The minimum number of token in a sentence is 1
- The maximum number of tokens in a sentence is 748

In [23]:
length_128 = df[df['tokens_length'] > 128]
length_256 = df[df['tokens_length'] > 256]

print(f"Number of sentences have larger then 128 tokens: {len(length_128)} ")
print(f"Number of sentences have larger then 256 tokens: {len(length_256)} ")

print(f'skills spotted in > 256 sentences: {length_256['num_skills'].sum()}')

max_idx = df['tokens_length'].idxmax()
print("Max length text:")
print(" ".join(df.loc[max_idx, 'tokens']))

Number of sentences have larger then 128 tokens: 46 
Number of sentences have larger then 256 tokens: 13 
skills spotted in > 256 sentences: 147
Max length text:
We’d like to meet someone with the following experience: HTML CSS JavaScript Creating reusable components across micro frontends Identifying site issues and customer journeys using analytics and MVT JavaScript fundamentals and modern syntax Modern JavaScript frameworks Bonus points for React/Redux Node.js for server-side rendering building micro services Building automated acceptance tests Bonus points for Cypress and WebDriver.io frameworks Writing unit tests following TDD Software craftsmanship using engineering fundamentals SOLID KISS DRY and YAGNI writing secure code Contributing to and maintaining an inner source GitHub repo CI / CD pipelines including Docker workflows HTTP and working with CDNs cloud technologies Bonus points for Azure and Kubernetes Building and maintaining user interfaces that support multiple devices 

### **NOTE**

- The number of sentences larger than 128 are moderate compared to the full dataset (59 sentences), but the number of skills spotted is more significant (147).
- Also, the largest length sentence contains a lot of skills and knowledge tags; since I will use BERT Transformer as one of the models, the model may cut off lengthy sentences. One good approach would be to split the sentence into smaller chunks, run the model and then combine them together

In [24]:
df_cleaned = df.drop(columns=['broken_span_skill', 'broken_span_knowledge'])
df_cleaned

,idx,tokens,tags_skill,tags_knowledge,source,split,num_skills,num_knowledge,has_overlap,tokens_length
0,1,"[Senior, QA, Engineer, (, m/f/d, ), <ORGANIZAT...","[O, O, O, O, O, O, O]","[O, O, O, O, O, O, O]",tech,train,0,0,False,7
1,1,"[<ADDRESS>, <ADDRESS>, <ADDRESS>, <ADDRESS>, <...","[O, O, O, O, O]","[O, O, O, O, O]",tech,train,0,0,False,5
2,1,"[Date, posted:, 2021-07-14]","[O, O, O]","[O, O, O]",tech,train,0,0,False,3
3,1,"[Likes:, 0, Dislikes:, 0, Love:, 0]","[O, O, O, O, O, O]","[O, O, O, O, O, O]",tech,train,0,0,False,6
4,1,"[Job, description:]","[O, O]","[O, O]",tech,train,0,0,False,2
...,...,...,...,...,...,...,...,...,...,...
11538,65,"[You, can, read, about, the, recruitment, proc...","[O, O, O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O, O, O]",house,test,0,0,False,10
11539,65,"[Assessment:, The, selected, applications, wil...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O]",house,test,0,0,False,14
11540,65,"[242, of, 13, March, 2012, on, the, Appointmen...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",house,test,0,0,False,20
11541,65,"[The, Assessment, Committee, makes, a, non-pri...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",house,test,0,0,False,31


In [25]:
import os

os.makedirs('../data', exist_ok = True)

df_cleaned.to_parquet('../data/df_cleaned.parquet')